# KHUDA 9기 ML세션
2026.02.19 (목)

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

In [2]:
# dataset repository path
url = "https://raw.githubusercontent.com/gaeng02/KHUDA-9th-ML/main/week5/dataset.csv"

In [3]:
df = pd.read_csv(url)
df.head()

,id,income_k,spend_score,age,visits_m
0,1,36.83,30.32,29.22,3.13
1,2,28.76,24.21,24.88,4.38
2,3,39.50,21.61,22.36,3.88
3,4,40.64,24.36,28.43,6.51
4,5,23.29,11.50,25.72,5.89


In [4]:
### answer init
answer = [0] * 5

## Task 0. 초기 설정
다음 설정에 맞게 데이터셋을 준비하세요. <br>
별도의 정답은 없지만, 완료되지 않은 경우 Task 수행 불가

- feature: `income_k, spend_score, age, visits_m`
- scaling: `StandardScaler()`

In [12]:
## Input Box

X = df[['income_k', 'spend_score', 'age', 'visits_m']].values
scaler = StandardScaler()
X = scaler.fit_transform(X)

X

array([[-1.01949107, -0.83619908, -0.51463934, -1.30431893],
       [-1.35284219, -1.07340664, -0.93679278, -0.84444852],
       [-0.90920019, -1.17434602, -1.18191414, -1.02839669],
       ...,
       [-1.15745795,  1.66942701,  0.6749774 , -2.43008167],
       [-1.33714536,  0.22638205,  1.2663813 , -2.24245455],
       [ 1.81006918,  1.11658977,  0.72944881,  1.30038704]])

## Task 1
Kmeans 모델에서 Silhouette을 이용하여 best k를 찾으세요. <br>
k의 범위는 2이상 8이하로, 가장 Silhouette Score가 큰 k를 `answer[1]`에 저장하세요. <br>
(만약, Silhouette score가 동일하다면, 작은 k를 선택) <br><br>

이때, `random_state = 219, n_init = 10`로 설정하세요.

In [13]:
## Input Box

best_k = None
best_score = 0.0

for i in range(2,9):
  kmeans = KMeans(n_clusters=i, init='k-means++', n_init=10, random_state=219).fit(X)
  average_score = silhouette_score(X, kmeans.labels_)
  if(average_score > best_score):
    best_score = average_score
    best_k = i


answer[1] = best_k
print(answer[1])

5


## Task 2
Task 1에서 구한 k 값으로 Kmeans를 학습하세요. <br>
클러스터 중 `spend_score` 평균이 가장 큰 클러스터의 sample 수를 `answer[2]`에 저장하세요. <br><br>

이때, `random_state = 219, n_init = 10`로 설정하세요.
<br><br>
[Hint] cluster별로 label을 추가해준 df_copy를 만들고 `groupby` 사용하기

In [22]:
## Input Box

count = 0

kmeans = KMeans(n_clusters=5, init='k-means++', random_state=219, n_init=10)
labels = kmeans.fit_predict(X)
df_copy = df.copy()
df_copy['cluster'] = labels
vip_cluster = df_copy.groupby('cluster')['spend_score'].mean().idxmax() # 해당 열의 최댓값이 있는 인덱스(클러스터 번호) 반환
count = (df_copy['cluster'] == vip_cluster).sum() # 해당 클러스터에 속하는 샘플 수 구하기

answer[2] = count
print(answer[2])


58


## Task 3
Task 2에서 학습한 Kmeans에 대해 `cluster_centers_`를 이용해, 서로 다른 두 중심 사이의 유클리드 거리 중 최대값을 구하세요. <br>
이때 최대 거리를 `answer[3]`에 저장하세요.

In [27]:
## Input Box

max_distance = 0.0
centers = kmeans.cluster_centers_

for i in range(len(centers)):
  for j in range(len(centers)):
    d = np.linalg.norm(centers[i] - centers[j]) # n차원 점 간의 유클리드 거리를 구하는 매서드
    if(d > max_distance):
      max_distance = d

answer[3] = max_distance
print(answer[3])

4.296828314241623


## Task 4
DBSCAN으로 군집을 만들고, 노이즈(클러스터에 속하지 못함)의 수를 구하세요. <br>
이때, 노이즈의 수를 `answer[4]`에 저장하세요.
<br><br>
이때, `eps = 0.5, min_samples = 5`로 설정하세요.

In [29]:
## Input Box

count = 0

dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X)
count = (dbscan_labels == -1).sum()


answer[4] = count
print(answer[4])

123


# 정답 확인!

In [30]:
print("=== 작성하신 정답 ===")
for i in range (1, 5) : print("Task " + str(i) + " :: " + str(answer[i]))

=== 작성하신 정답 ===
Task 1 :: 5
Task 2 :: 58
Task 3 :: 4.296828314241623
Task 4 :: 123


In [31]:
### 정답 채점 코드
import hashlib

ANSWER_URL = "https://raw.githubusercontent.com/gaeng02/KHUDA-9th-ML/main/week5/answer.csv"

def md5 (s) : return hashlib.md5(s.encode("utf-8")).hexdigest()

ans_df = pd.read_csv(ANSWER_URL)
gt = {int(r["task"]): str(r["answer"]).strip().lower()
      for _, r in ans_df.iterrows()}

wrong = []

for i in range(1, 5) :
    user_answer = "" if answer[i] is None else md5(str(answer[i]).strip().lower())
    if (user_answer != gt.get(i, "")) : wrong.append(i)

print("탈출!" if not wrong else f"틀린번호 : {wrong}")

탈출!
